# Environmental own-knockout: ghost + hazard-off (poison)

The environmental cell's positive control. Full oversight removal (`ghost_keeps_bonus=False`) **plus** `hazard_off_after=1000` gating the physical -4 penalty (`bwj` poison-land) in phase 2. Pairs against the existing poison-ghost result (`extinct3_cleanghost.csv` cond (0,)): ghost-alone **persists** (the hazard still selects avoidance), ghost+hazard-off **decays** (nothing left to ground it). That pairing IS the environmental-grounding proof. 5 seeds, matches the cleanghost grid.

## 0. Setup (pure Python; leaves the pod's jax untouched)

In [ ]:
try:
    import flax, optax
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'flax', 'optax'])
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
import jax
print('jax', jax.__version__, '| devices:', jax.devices())
if jax.devices()[0].platform != 'gpu':
    print('WARNING: NOT on GPU -> run_extinction will refuse a real run. Rebind & re-run.')


## 1. Write sources (base64 -> files; incl. the oracle-diffed coordination/hazard flags)

In [ ]:
import base64, pathlib
_SRC = {
  'berryworld.py': '...',
  'berryworld_jax.py': '...',
  'train_jax.py': '...',
  'run_sweep.py': 'IiIiCnJ1bl9zd2VlcC5weSAtLSB0aGUgZmFpdGhmdWwtc2NhbGUgcG9wdWxhdGlvbiBzd2VlcCwgZHJpdmVyIGZvciBHUFUgKENvbGFiKS4KCkZvciBlYWNoIChOLCBjb25kaXRpb24pIGNlbGwgaXQgdm1hcHMgdGhlIEpBWCBJUFBPIHRyYWluIG92ZXIgc2VlZHMgYW5kIGxvZ3MKcGVyLXVwZGF0ZSBtZXRyaWNzLiBUaGUgcXVlc3Rpb24gdGhpcyBhbnN3ZXJzOiBkb2VzIGVuZm9yY2VtZW50IChzZWxlY3Rpdml0eT4xKQpmaXJlIHdoZW4gdGhlIHBvcHVsYXRpb24gZ3Jvd3MsIGdpdmVuIGVub3VnaCB0cmFpbmluZz8KCkxvY2FsIChDUFUpIHNtb2tlOiAgcHl0aG9uIHJ1bl9zd2VlcC5weSAtLXNtb2tlCkZhaXRoZnVsIChHUFUpOiAgICAgaW1wb3J0ZWQgZnJvbSB0aGUgQ29sYWIgbm90ZWJvb2sgd2l0aCBhIGxhcmdlIGhwLgoKTk9URSAvIGhvbmVzdCBsaW1pdGF0aW9ucyBvZiB0aGlzIGRpYWdub3N0aWMgdmVyc2lvbjoKICAqIEFsbCBOIGFnZW50cyBwbGF5IGV2ZXJ5IGVwaXNvZGUgKEtvc3RlcidzIDgtb2YtMTIgcGVyLWVwaXNvZGUgc2FtcGxpbmcgaXMgYQogICAgcmVmaW5lbWVudCBub3QgeWV0IGltcGxlbWVudGVkIC0tIHRoaXMgdGVzdHMgcG9wdWxhdGlvbiBTSVpFLCB0aGUgcHJpbWFyeQogICAgbGV2ZXIpLgogICogcGF0Y2hfbWFzayBpcyBmaXhlZCBwZXIgKE4sY29uZGl0aW9uKSBjZWxsIGFjcm9zcyBzZWVkczsgc2VlZCB2YXJpZXMgbmV0CiAgICBpbml0ICsgcm9sbG91dCArIHJlc2V0cy4gUGVyLXNlZWQgcGF0Y2hlcyBhcmUgdGhlIHJpZ29yb3VzIHZlcnNpb24sIGxhdGVyLgoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KaW1wb3J0IGpheAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRyYWluX2pheCBhcyBUCmltcG9ydCBiZXJyeXdvcmxkX2pheCBhcyBid2oKCgpkZWYgZW52X3ZhcmlhbnQocG9pc29uX2RlbGF5PTI1LCByX3phcF9ib251cz0wLjAsCiAgICAgICAgICAgICAgICBlcGlzb2RlX2xlbj0zMDAsIHphcF9yZW1vdmFsX3N0ZXBzPTI1LCBvYnNlcnZlX3BlbmRpbmc9RmFsc2UsCiAgICAgICAgICAgICAgICBib251c19yZXF1aXJlc19tYXJrPUZhbHNlLCBhdXRvX3RhcmdldD1GYWxzZSwgY196YXBwZWQ9Mi4wLAogICAgICAgICAgICAgICAgbl9iZXJyeV90eXBlcz0yLCBncmlkPTE1LCBnaG9zdF9rZWVwc19ib251cz1UcnVlLAogICAgICAgICAgICAgICAgY29udmVyZ2VudF9iZXJyeT1Ob25lLCBjb29yZF9rPTAuMCwgY29vcmRfYT0xLjUpOgogICAgcmV0dXJuIGRpY3QocG9pc29uX2RlbGF5PXBvaXNvbl9kZWxheSwgcl96YXBfYm9udXM9cl96YXBfYm9udXMsCiAgICAgICAgICAgICAgICBlcGlzb2RlX2xlbj1lcGlzb2RlX2xlbiwgemFwX3JlbW92YWxfc3RlcHM9emFwX3JlbW92YWxfc3RlcHMsCiAgICAgICAgICAgICAgICBvYnNlcnZlX3BlbmRpbmc9b2JzZXJ2ZV9wZW5kaW5nLAogICAgICAgICAgICAgICAgYm9udXNfcmVxdWlyZXNfbWFyaz1ib251c19yZXF1aXJlc19tYXJrLAogICAgICAgICAgICAgICAgYXV0b190YXJnZXQ9YXV0b190YXJnZXQsIGNfemFwcGVkPWNfemFwcGVkLAogICAgICAgICAgICAgICAgbl9iZXJyeV90eXBlcz1uX2JlcnJ5X3R5cGVzLCBncmlkPWdyaWQsCiAgICAgICAgICAgICAgICBnaG9zdF9rZWVwc19ib251cz1naG9zdF9rZWVwc19ib251cywKICAgICAgICAgICAgICAgIGNvbnZlcmdlbnRfYmVycnk9Y29udmVyZ2VudF9iZXJyeSwgY29vcmRfaz1jb29yZF9rLCBjb29yZF9hPWNvb3JkX2EpCgoKZGVmIHJ1bl9zd2VlcChuX2xpc3QsIGNvbmRpdGlvbnMsIG5fc2VlZHMsIGhwLCBlbnZfbGlzdCwgb3V0X2Nzdj0ic3dlZXAuY3N2IiwKICAgICAgICAgICAgICB2bWFwX3NlZWRzPUZhbHNlKToKICAgICIiIlN3ZWVwIG92ZXIgTiB4IGVudl92YXJpYW50IHggY29uZGl0aW9uLiBFYWNoIGVudl92YXJpYW50IGNhcnJpZXMgaXRzIG93bgogICAgcG9pc29uX2RlbGF5IChkaWZmaWN1bHR5KSBhbmQgcl96YXBfYm9udXMgKGVuZm9yY2VtZW50IGluY2VudGl2ZSksIGJvdGgKICAgIGxvZ2dlZCBwZXIgcm93IHNvIHRoZSBEIGFuZCBib251cyBheGVzIGFyZSBhbmFseXNhYmxlLiIiIgogICAgaWYgaXNpbnN0YW5jZShlbnZfbGlzdCwgZGljdCk6ICAgICAgICAgICAgICAgICAjIGFsbG93IGEgc2luZ2xlIGVudgogICAgICAgIGVudl9saXN0ID0gW2Vudl9saXN0XQogICAgIyBHdWFyZDogcmVmdXNlIGEgcmVhbCBzd2VlcCBvbiBDUFUuIFRoZSB3aG9sZSBwb2ludCBvZiB0aGUgSkFYIHBpdm90IGlzCiAgICAjIEdQVTsgYSBDUFUgcnVuIGhlcmUgbWVhbnMgdGhlIG5vdGVib29rIGlzIGV4ZWN1dGluZyBvbiB0aGUgbGFwdG9wLCBub3QKICAgICMgQ29sYWIuIEFib3J0IGluIDwxcyBpbnN0ZWFkIG9mIGdyaW5kaW5nIGZvciBob3Vycy4gVGlueSBkZXZpY2Utc2FuaXR5CiAgICAjIHJ1bnMgKHVwZGF0ZXMqbnVtX2VudnMgPD0gMTAwMCkgYXJlIHN0aWxsIGFsbG93ZWQgb24gQ1BVLgogICAgaWYgbm90IGFueShkLnBsYXRmb3JtID09ICJncHUiIGZvciBkIGluIGpheC5kZXZpY2VzKCkpIFwKICAgICAgICAgICAgYW5kIGhwWyJ1cGRhdGVzIl0gKiBocFsibnVtX2VudnMiXSA+IDEwMDA6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICBmInJ1bl9zd2VlcCBhYm9ydGVkOiBOTyBHUFUgKGpheC5kZXZpY2VzKCk9e2pheC5kZXZpY2VzKCl9KS4gIgogICAgICAgICAgICBmIlRoaXMgaXMgYSBDUFUgcnVudGltZSAtLSB5b3VyIGxhcHRvcCwgbm90IENvbGFiIEdQVSAtLSBhbmQgIgogICAgICAgICAgICBmInVwZGF0ZXMqbnVtX2VudnM9e2hwWyd1cGRhdGVzJ10gKiBocFsnbnVtX2VudnMnXX0gaXMgYSByZWFsICIKICAgICAgICAgICAgZiJzd2VlcCAoaG91cnMgb24gQ1BVKS4gT24gQ29sYWI6IFJ1bnRpbWUgLT4gQ2hhbmdlIHJ1bnRpbWUgdHlwZSAiCiAgICAgICAgICAgIGYiLT4gR1BVLCB0aGVuIFJ1biBhbGwuIikKICAgICMgQ3Jhc2gtc2FmZTogd3JpdGUgZWFjaCBjZWxsJ3Mgcm93cyB0aGUgaW5zdGFudCBpdCBmaW5pc2hlcyBhbmQgZmx1c2gsIHNvIGFuCiAgICAjIGludGVycnVwdCAvIENvbGFiIHRpbWVvdXQgLyBwcmVlbXB0aW9uIGtlZXBzIGV2ZXJ5IENPTVBMRVRFRCBjZWxsIGluc3RlYWQgb2YKICAgICMgbG9zaW5nIHRoZSB3aG9sZSBydW4gKHRoZSBlYXJsaWVyIGZhaWx1cmUgbW9kZSAtLSBzZWUgcHJvamVjdCBub3RlcyBvbgogICAgIyBtYWtpbmcgb3V0cHV0cyByZXN1bWFibGUpLiBgcm93c2AgaXMgc3RpbGwgYWNjdW11bGF0ZWQgZm9yIHRoZSByZXR1cm4gdmFsdWUuCiAgICBuYnQgPSBlbnZfbGlzdFswXS5nZXQoIm5fYmVycnlfdHlwZXMiLCAyKSAgICMgcGVyLXR5cGUgZWF0L2VuYyBjb2x1bW5zCiAgICBmaWVsZG5hbWVzID0gKFsiTiIsICJEIiwgImJvbnVzIiwgImNvbmRpdGlvbiIsICJzZWVkIiwgInVwZGF0ZSJdCiAgICAgICAgICAgICAgICAgICsgW2YiZWF0e3R9IiBmb3IgdCBpbiByYW5nZShuYnQpXSArIFtmImVuY3t0fSIgZm9yIHQgaW4gcmFuZ2UobmJ0KV0KICAgICAgICAgICAgICAgICAgKyBbInNlbGVjdGl2aXR5IiwgInByZXZhbGVuY2UiLCAiemFwcyIsICJyZXQiXSkKICAgIHJvd3MgPSBbXQogICAgIyBSZXByb2R1Y2liaWxpdHk6IGR1bXAgdGhlIEZVTEwgcmVjaXBlIG5leHQgdG8gdGhlIENTVi4gVGhlIHBpbG90IHByb3ZlZCB0aGUKICAgICMgQ1NWJ3MgNCBjb25maWcgY29sdW1ucyAoTi9EL2JvbnVzL3NlZWQpIFVOREVSLXNwZWNpZnkgYSBydW4gLS0gbnVtX2VudnMgd2FzCiAgICAjIHRoZSBkaWZmZXJlbmNlIHRoYXQgZmxpcHBlZCB0aGUgd2hvbGUgcmVzdWx0IC0tIHNvIHBlcnNpc3QgaHAgKyBlbnYgKyBzZWVkcy4KICAgIHdpdGggb3BlbihvdXRfY3N2ICsgIi5jb25maWcuanNvbiIsICJ3IikgYXMgY2Y6CiAgICAgICAganNvbi5kdW1wKHsiaHAiOiBocCwgIm5fc2VlZHMiOiBuX3NlZWRzLAogICAgICAgICAgICAgICAgICAgImNvbmRpdGlvbnMiOiBbbGlzdChjKSBmb3IgYyBpbiBjb25kaXRpb25zXSwKICAgICAgICAgICAgICAgICAgICJlbnZfbGlzdCI6IFtkaWN0KGUpIGZvciBlIGluIGVudl9saXN0XSwKICAgICAgICAgICAgICAgICAgICJuX2xpc3QiOiBsaXN0KG5fbGlzdCl9LCBjZiwgaW5kZW50PTIpCiAgICBmID0gb3BlbihvdXRfY3N2LCAidyIsIG5ld2xpbmU9IiIpCiAgICB3ID0gY3N2LkRpY3RXcml0ZXIoZiwgZmllbGRuYW1lcz1maWVsZG5hbWVzKQogICAgdy53cml0ZWhlYWRlcigpOyBmLmZsdXNoKCkKICAgIHRyeToKICAgICAgICBmb3IgTiBpbiBuX2xpc3Q6CiAgICAgICAgICAgIGZvciBlbnYgaW4gZW52X2xpc3Q6CiAgICAgICAgICAgICAgICBmb3IgbWFya2VkIGluIGNvbmRpdGlvbnM6CiAgICAgICAgICAgICAgICAgICAgY2ZnID0gYndqLkpDZmcoCiAgICAgICAgICAgICAgICAgICAgICAgIG5fYWdlbnRzPU4sIGVwaXNvZGVfbGVuPWVudlsiZXBpc29kZV9sZW4iXSwKICAgICAgICAgICAgICAgICAgICAgICAgcG9pc29uX2RlbGF5PWVudlsicG9pc29uX2RlbGF5Il0sCiAgICAgICAgICAgICAgICAgICAgICAgIHphcF9yZW1vdmFsX3N0ZXBzPWVudlsiemFwX3JlbW92YWxfc3RlcHMiXSwKICAgICAgICAgICAgICAgICAgICAgICAgcl96YXBfYm9udXM9ZW52WyJyX3phcF9ib251cyJdLAogICAgICAgICAgICAgICAgICAgICAgICBvYnNlcnZlX3BlbmRpbmc9ZW52LmdldCgib2JzZXJ2ZV9wZW5kaW5nIiwgRmFsc2UpLAogICAgICAgICAgICAgICAgICAgICAgICBib251c19yZXF1aXJlc19tYXJrPWVudi5nZXQoImJvbnVzX3JlcXVpcmVzX21hcmsiLCBGYWxzZSksCiAgICAgICAgICAgICAgICAgICAgICAgIGF1dG9fdGFyZ2V0PWVudi5nZXQoImF1dG9fdGFyZ2V0IiwgRmFsc2UpLAogICAgICAgICAgICAgICAgICAgICAgICBjX3phcHBlZD1lbnYuZ2V0KCJjX3phcHBlZCIsIDIuMCksCiAgICAgICAgICAgICAgICAgICAgICAgIGdyaWQ9ZW52LmdldCgiZ3JpZCIsIDE1KSwKICAgICAgICAgICAgICAgICAgICAgICAgbl9iZXJyeV90eXBlcz1lbnYuZ2V0KCJuX2JlcnJ5X3R5cGVzIiwgMiksCiAgICAgICAgICAgICAgICAgICAgICAgIG1hcmtlZF9tYXNrPXR1cGxlKHQgaW4gbWFya2VkIGZvciB0IGluIHJhbmdlKGVudi5nZXQoIm5fYmVycnlfdHlwZXMiLCAyKSkpKQogICAgICAgICAgICAgICAgICAgIHBtID0gVC5idWlsZF9wYXRjaF9tYXNrKG1hcmtlZCwgTiwgbl9iZXJyeV90eXBlcz1lbnYuZ2V0KCJuX2JlcnJ5X3R5cGVzIiwgMiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZ3JpZD1lbnYuZ2V0KCJncmlkIiwgMTUpKQogICAgICAgICAgICAgICAgICAgIHRyYWluMSA9IFQubWFrZV90cmFpbihjZmcsIHBtLCBocCkKICAgICAgICAgICAgICAgICAgICBrZXlzID0gamF4LnJhbmRvbS5zcGxpdChqYXgucmFuZG9tLlBSTkdLZXkoMCksIG5fc2VlZHMpCiAgICAgICAgICAgICAgICAgICAgIyBTZWVkcyBhcmUgSU5ERVBFTkRFTlQgKG5vIGNyb3NzLXNlZWQgcmVkdWN0aW9uKSwgc28gdm1hcHBpbmcKICAgICAgICAgICAgICAgICAgICAjIHRoZW0gb25seSBwYXJhbGxlbGlzZXMgLS0gaXQgZG9lcyBub3QgY2hhbmdlIHBlci1zZWVkIG1hdGgsCiAgICAgICAgICAgICAgICAgICAgIyBidXQgaXQgaG9sZHMgYWxsIG5fc2VlZHMgdHJhaW5pbmcgZ3JhcGhzIG9uIHRoZSBHUFUgYXQgb25jZQogICAgICAgICAgICAgICAgICAgICMgKH5uX3NlZWRzIHggbWVtb3J5LCB0aGUgT09NIGF0IGxhcmdlIG51bV9lbnZzKS4gRGVmYXVsdCBydW5zCiAgICAgICAgICAgICAgICAgICAgIyB0aGVtIFNFUVVFTlRJQUxMWTogb25lIHNlZWQncyBncmFwaCByZXNpZGVudCBhdCBhIHRpbWUsIGVhY2gKICAgICAgICAgICAgICAgICAgICAjIHB1bGxlZCB0byBob3N0IHNvIGl0cyBkZXZpY2UgYnVmZmVycyBmcmVlIGJlZm9yZSB0aGUgbmV4dC4KICAgICAgICAgICAgICAgICAgICAjIHZtYXBfc2VlZHM9VHJ1ZSByZXN0b3JlcyB0aGUgcGFyYWxsZWwgcGF0aCAobmVlZHMgdGhlIFZSQU07CiAgICAgICAgICAgICAgICAgICAgIyBpdHMgWExBIGNvZGVnZW4gZGlmZmVycywgc28ga2VlcCBpdCBmb3IgZXhhY3Qta29zdGVyIHJlcHJvKS4KICAgICAgICAgICAgICAgICAgICBpZiB2bWFwX3NlZWRzOgogICAgICAgICAgICAgICAgICAgICAgICBtZm4gPSBqYXguaml0KGpheC52bWFwKGxhbWJkYSBrOiB0cmFpbjEoaylbMV0pKQogICAgICAgICAgICAgICAgICAgICAgICBtbSA9IGpheC5ibG9ja191bnRpbF9yZWFkeShtZm4oa2V5cykpICAgICAgICAgICMgZWFjaCAoUywgVSkKICAgICAgICAgICAgICAgICAgICAgICAgbSA9IHtrazogbnAuYXNhcnJheSh2dikgZm9yIGtrLCB2diBpbiBtbS5pdGVtcygpfQogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHNlZWRfbWV0cmljcyA9IGpheC5qaXQobGFtYmRhIGs6IHRyYWluMShrKVsxXSkKICAgICAgICAgICAgICAgICAgICAgICAgcGVyX3NlZWQgPSBbXQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgayBpbiBrZXlzOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgbWsgPSBqYXguYmxvY2tfdW50aWxfcmVhZHkoc2VlZF9tZXRyaWNzKGspKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgcGVyX3NlZWQuYXBwZW5kKHtrazogbnAuYXNhcnJheSh2dikKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGtrLCB2diBpbiBtay5pdGVtcygpfSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRlbCBtawogICAgICAgICAgICAgICAgICAgICAgICBtID0ge2trOiBucC5zdGFjayhbcHNba2tdIGZvciBwcyBpbiBwZXJfc2VlZF0sIDApICAjIChTLCBVKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBrayBpbiBwZXJfc2VlZFswXX0KICAgICAgICAgICAgICAgICAgICBjb25kID0gIiIuam9pbihzdHIoYikgZm9yIGIgaW4gbWFya2VkKSBvciAibm9uZSIKICAgICAgICAgICAgICAgICAgICBVID0gbVsiZWF0MCJdLnNoYXBlWzFdCiAgICAgICAgICAgICAgICAgICAgZSA9IG1bImVhdDAiXTsgc2VsID0gbVsic2VsZWN0aXZpdHkiXQogICAgICAgICAgICAgICAgICAgIHByaW50KGYiTj17TjoyZH0gRD17ZW52Wydwb2lzb25fZGVsYXknXToyZH0gIgogICAgICAgICAgICAgICAgICAgICAgICAgIGYiYm9udXM9e2Vudlsncl96YXBfYm9udXMnXTouMWZ9IHtjb25kOjRzfSAgIgogICAgICAgICAgICAgICAgICAgICAgICAgIGYiZWF0MCB7ZVs6LCA6NV0ubWVhbigpOjQuMGZ9LT57ZVs6LCAtNTpdLm1lYW4oKTo0LjBmfSIKICAgICAgICAgICAgICAgICAgICAgICAgICBmIiAgc2VsIHtzZWxbOiwgLTU6XS5tZWFuKCk6LjJmfSAgKG49e25fc2VlZHN9KSIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICAgICAgY2VsbF9yb3dzID0gW2RpY3QoCiAgICAgICAgICAgICAgICAgICAgICAgIE49TiwgRD1lbnZbInBvaXNvbl9kZWxheSJdLAogICAgICAgICAgICAgICAgICAgICAgICBib251cz1lbnZbInJfemFwX2JvbnVzIl0sIGNvbmRpdGlvbj1jb25kLAogICAgICAgICAgICAgICAgICAgICAgICBzZWVkPXMsIHVwZGF0ZT11LAogICAgICAgICAgICAgICAgICAgICAgICBzZWxlY3Rpdml0eT1mbG9hdChtWyJzZWxlY3Rpdml0eSJdW3MsIHVdKSwKICAgICAgICAgICAgICAgICAgICAgICAgcHJldmFsZW5jZT1mbG9hdChtWyJwcmV2YWxlbmNlIl1bcywgdV0pLAogICAgICAgICAgICAgICAgICAgICAgICB6YXBzPWZsb2F0KG1bInphcHMiXVtzLCB1XSksIHJldD1mbG9hdChtWyJyZXQiXVtzLCB1XSksCiAgICAgICAgICAgICAgICAgICAgICAgICoqe2YiZWF0e3R9IjogZmxvYXQobVtmImVhdHt0fSJdW3MsIHVdKSBmb3IgdCBpbiByYW5nZShuYnQpfSwKICAgICAgICAgICAgICAgICAgICAgICAgKip7ZiJlbmN7dH0iOiBmbG9hdChtW2YiZW5je3R9Il1bcywgdV0pIGZvciB0IGluIHJhbmdlKG5idCl9KQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgcyBpbiByYW5nZShuX3NlZWRzKSBmb3IgdSBpbiByYW5nZShVKV0KICAgICAgICAgICAgICAgICAgICB3LndyaXRlcm93cyhjZWxsX3Jvd3MpOyBmLmZsdXNoKCkgICAgICAgICAgIyBwZXJzaXN0IHRoaXMgY2VsbAogICAgICAgICAgICAgICAgICAgIHJvd3MuZXh0ZW5kKGNlbGxfcm93cykKICAgIGZpbmFsbHk6CiAgICAgICAgZi5jbG9zZSgpCiAgICBwcmludChmIndyb3RlIHtsZW4ocm93cyl9IHJvd3MgLT4ge291dF9jc3Z9IiwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiByb3dzCgoKRU5WID0gZW52X3ZhcmlhbnQoKSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBEPTI1LCBib251cz0wIGRlZmF1bHQKCiMgRmFpdGhmdWwtc2NhbGUgaHlwZXJwYXJhbWV0ZXJzIGZvciBHUFUuIHN0ZXBzID0gdXBkYXRlcyAqIG51bV9lbnZzICogZXBpc29kZV9sZW4uCiMgdXBkYXRlcz0xNTAwLCBudW1fZW52cz0yNTYgLT4gfjEuMTVlOCBzdGVwcy9ydW4gKEtvc3RlciByZWdpbWUgaXMgMi00ZTgpLgpGQUlUSEZVTF9IUCA9IGRpY3QoaGlkZGVuPTY0LCBscj0zZS00LCBnYW1tYT0wLjk5LCBsYW09MC45NSwgY2xpcD0wLjIsCiAgICAgICAgICAgICAgICAgICBlcG9jaHM9MywgZW50PTAuMDEsIHZmPTAuNSwgbWF4X2dyYWQ9MC41LAogICAgICAgICAgICAgICAgICAgbnVtX2VudnM9MjU2LCB1cGRhdGVzPTE1MDApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNtb2tlIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwgaGVscD0idGlueSBDUFUgcnVuIikKICAgIGEgPSBhcC5wYXJzZV9hcmdzKCkKICAgIGlmIGEuc21va2U6CiAgICAgICAgaHAgPSBkaWN0KEZBSVRIRlVMX0hQKTsgaHBbIm51bV9lbnZzIl0gPSA4OyBocFsidXBkYXRlcyJdID0gMjAKICAgICAgICBlbnZzID0gW2Vudl92YXJpYW50KHBvaXNvbl9kZWxheT1kKSBmb3IgZCBpbiAoMjUsIDc1KV0gICAgICMgRCBheGlzCiAgICAgICAgcnVuX3N3ZWVwKFsxMl0sIFsoKSwgKDAsKSwgKDAsIDEpXSwgbl9zZWVkcz0yLCBocD1ocCwgZW52X2xpc3Q9ZW52cywKICAgICAgICAgICAgICAgICAgb3V0X2Nzdj0ic3dlZXBfc21va2UuY3N2IikKICAgIGVsc2U6CiAgICAgICAgcnVuX3N3ZWVwKFsxMCwgMTJdLCBbKCksICgwLCksICgwLCAxKV0sIG5fc2VlZHM9OCwKICAgICAgICAgICAgICAgICAgaHA9RkFJVEhGVUxfSFAsIGVudl9saXN0PVtFTlZdLCBvdXRfY3N2PSJzd2VlcC5jc3YiKQo=',
  'extinct.py': '...',
}
for _n, _b in _SRC.items(): pathlib.Path(_n).write_bytes(base64.b64decode(_b))
print('wrote sources:', list(_SRC))


## 2. Run  (expect `>>> RUN START`, per-condition/per-chunk lines, then `>>> RUN DONE`)

In [ ]:
print('>>> RUN START', flush=True)
import run_sweep as R, extinct
ENV = R.env_variant(poison_delay=100, r_zap_bonus=8.75, episode_len=300,
                    zap_removal_steps=25, bonus_requires_mark=True,
                    c_zapped=2.0, n_berry_types=3, grid=22, ghost_keeps_bonus=False)
hp = dict(R.FAITHFUL_HP); hp['updates']=1600
extinct.run_extinction([(0,)], ENV, hp, n_install=1000, n_seeds=5,
                       hazard_off_after=1000, vmap_seeds=5,
                       out_csv='poison_hazardoff.csv')
extinct.analyze('poison_hazardoff.csv')
print('>>> RUN DONE', flush=True)
